In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:
###Define Thresholds

rare_species_abundance_threshold = 6.5
expressed_gene_reads_threshold = 2000
number_of_samples_expressed_in_threshold = 21

In [ ]:
### Note we are not removing the low quality samples as that is transcriptomic quailty, this is microbiome only analysis

metadata = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Plates_1_to_4_metadata_merged_luke.csv",
    index_col=0,
)
metadata = metadata.drop(
    columns=[
        "arb.sort",
        "sample-id",
        "Ambiguous Unstranded",
        "Ambiguous Forward",
        "Multimapping",
        "Unmapped Over Mapped",
    ]
)
metadata["Date and Time"] = metadata["date"] + " " + metadata["time"]
luke_time_data_format = "%-m/%-d/%y %-H:%-M"
metadata["Date and Time"] = pd.to_datetime(
    metadata["Date and Time"], format=luke_time_data_format
)


unusable_samples = [
    "A2450525897_n01_LICRNA01_A06",
    "A2450525897_n01_LICRNA01_A08",
    "A2450525897_n01_LICRNA01_C11",
    "A2450525897_n01_LICRNA01_D11",
    "A2450525897_n01_LICRNA01_D12",
    "A2449446903_n01_LICRNA02_F01",
    "B2449500127_n01_LICRNA04_A04",
    "B2449500127_n01_LICRNA04_A06",
    "B2449500127_n01_LICRNA04_A07",
]
borderline_unusable = [
    "A2450525897_n01_LICRNA01_F03",
    "A2450525897_n01_LICRNA01_G07",
    "A2449446903_n01_LICRNA02_A04",
    "A2449446903_n01_LICRNA02_H10",
    "B2449500127_n01_LICRNA04_H02",
    "B2449500127_n01_LICRNA04_H11",
]
all_unsable = borderline_unusable + unusable_samples

# trimmed_metadata = metadata.drop(index=all_unsable)

trimmed_metadata = metadata.sort_index()
microbiome_abundance = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/Microbiome/lic2024_16S_rab.csv"
)
long_term_microbiome = microbiome_abundance.merge(
    metadata[["sampID", "Experiment Type"]], left_on="plantID", right_on="sampID"
)
species_abundance_table = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/species_core_results.csv"
)
species_abundance_table["rare"] = False
species_abundance_table.loc[
    species_abundance_table["n_samples_present"] <= (len(trimmed_metadata) * 0.1),
    "rare",
] = True
list_of_rare_species = species_abundance_table.loc[
    species_abundance_table["rare"] == True, "Species"
].tolist()
list_of_core_species = species_abundance_table.loc[
    species_abundance_table["core"] == True, "Species"
].tolist()

# For each sample, check if it has any rare species or is lacking any core species
rare_species_set = set(list_of_rare_species)
core_species_set = set(list_of_core_species)

# Get species present for each sample (using sampID from long_term_microbiome)
sample_species = long_term_microbiome.groupby("sampID")["Species"].apply(set)

# Check if sample has any rare species
has_rare_species = sample_species.apply(lambda x: len(x & rare_species_set) > 0)

# Check if sample is lacking any core species (missing at least one core species)
lacking_core_species = sample_species.apply(lambda x: not core_species_set.issubset(x))

# Total relative abundance of rare species per sample (sum of AbundR100 over rare species)
rare_abund_by_sample = (
    long_term_microbiome.loc[
        long_term_microbiome["Species"].isin(rare_species_set),
        ["sampID", "AbundR100"],
    ]
    .assign(
        AbundR100=lambda d: pd.to_numeric(d["AbundR100"], errors="coerce").fillna(
            0.0
        )  ##Junky column that just makes sure that its numeric
    )
    .groupby("sampID", sort=False)["AbundR100"]
    .sum()
    .rename("rare_species_total_abundR100")
)

# Add columns to metadata using the sampID column (not the index)
trimmed_metadata["has_rare_species"] = (
    trimmed_metadata["sampID"].map(has_rare_species).fillna(False)
)
trimmed_metadata["lacking_core_species"] = (
    trimmed_metadata["sampID"].map(lacking_core_species).fillna(True)
)
trimmed_metadata["rare_species_total_abundR100"] = (
    trimmed_metadata["sampID"].map(rare_abund_by_sample).fillna(0.0)
)

trimmed_metadata[
    [
        "sampID",
        "has_rare_species",
        "lacking_core_species",
        "rare_species_total_abundR100",
    ]
]

/var/folders/nk/6xkk9sgn1pz4ff1b36sfq3y40000gt/T/ipykernel_7290/2275921812.py:99: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trimmed_metadata["sampID"].map(has_rare_species).fillna(False)
/var/folders/nk/6xkk9sgn1pz4ff1b36sfq3y40000gt/T/ipykernel_7290/2275921812.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trimmed_metadata["sampID"].map(lacking_core_species).fillna(True)


,sampID,has_rare_species,lacking_core_species,rare_species_total_abundR100
filename,,,,
A2449446903_n01_LICRNA02_A01,LIC157,True,False,13.5
A2449446903_n01_LICRNA02_A02,LIC169,True,True,0.7
A2449446903_n01_LICRNA02_A03,LIC193,True,False,1.7
A2449446903_n01_LICRNA02_A04,LIC205,True,False,4.0
A2449446903_n01_LICRNA02_A05,LIC217,True,False,1.0
...,...,...,...,...
B250508004_n01_LICRNA03_H08,LIC428,False,True,0.0
B250508004_n01_LICRNA03_H09,LIC440,False,False,0.0
B250508004_n01_LICRNA03_H10,LIC452,True,True,1.0


In [ ]:
start_time = dt.datetime(2023, 10, 1)
end_time = dt.datetime(2024, 4, 30)
luke_apt_point = meteostat.Point(40.73005, -73.99450)
luke_hourly_data = meteostat.Hourly(luke_apt_point, start_time, end_time).fetch()
luke_rain_events = luke_hourly_data.loc[luke_hourly_data["prcp"] >= 0.2]
luke_rain_events.index = pd.to_datetime(luke_rain_events.index)
list_of_time_since_rain = []
for time in long_term_metadata["Date and Time"].to_list():
    time_since_all_rain = time - luke_rain_events.index
    rain_events_in_past = luke_rain_events.loc[
        time_since_all_rain >= dt.timedelta(seconds=0)
    ]
    time_of_last_rain = rain_events_in_past.index.max()
    time_since_last_rain = time - time_of_last_rain
    list_of_time_since_rain.append(time_since_last_rain)
long_term_metadata["Time Since Rain"] = list_of_time_since_rain
timepoint_time_since_rain = long_term_metadata[
    ["timepoint", "Time Since Rain"]
].drop_duplicates()
timepoint_time_since_rain = timepoint_time_since_rain.sort_values(by="timepoint")
timepoint_time_since_rain["Hours Since Rain"] = timepoint_time_since_rain[
    "Time Since Rain"
] / pd.Timedelta(hours=1)
timepoint_time_since_rain = timepoint_time_since_rain.set_index("timepoint", drop=True)
timepoint_time_since_rain